# Hangman (VS Code / Jupyter)
Run **cell 1** once (installs ipywidgets), then run **cell 2** to play.

In [ ]:
%pip install -q ipywidgets

In [ ]:
import random
import string
import ipywidgets as widgets
from IPython.display import HTML, display

WORDS = [
    "python", "hangman", "computer", "keyboard", "science", "developer",
    "algorithm", "variable", "function", "database", "software", "hardware",
    "internet", "compiler", "debugging", "framework", "javascript", "recursion",
    "elephant", "giraffe", "penguin", "dolphin", "kangaroo", "crocodile",
    "butterfly", "hedgehog", "octopus", "flamingo", "cheetah", "raccoon",
    "spaghetti", "chocolate", "pineapple", "sandwich", "avocado", "croissant",
    "australia", "argentina", "portugal", "thailand", "singapore", "himalaya",
    "umbrella", "telescope", "volcano", "interpreter", "symphony", "adventure",
    "mystery", "treasure", "wizard", "dinosaur", "astronaut", "galaxy",
]
MAX_WRONG = 6
state = {}

# Fixed colours so the game looks the same in VS Code light and dark themes
TEXT = "color: #222; font-family: sans-serif;"

title_html = widgets.HTML(f"<h1 style='{TEXT} margin: 0;'>HANGMAN GAME</h1>")
drawing = widgets.HTML(layout=widgets.Layout(margin="15px 0 0 0"))
word_html = widgets.HTML(layout=widgets.Layout(margin="10px 0 0 0"))
status_html = widgets.HTML()
new_btn = widgets.Button(
    description="New Game",
    button_style="primary",
    layout=widgets.Layout(width="125px", height="35px", margin="25px 0 5px 0"),
)
letter_buttons = {}


def draw(wrong):
    parts = [
        '<circle cx="150" cy="75" r="20" />',
        '<line x1="150" y1="95" x2="150" y2="170" />',
        '<line x1="150" y1="110" x2="120" y2="145" />',
        '<line x1="150" y1="110" x2="180" y2="145" />',
        '<line x1="150" y1="170" x2="125" y2="215" />',
        '<line x1="150" y1="170" x2="175" y2="215" />',
    ]
    body = "".join(parts[:wrong])
    return f"""
    <div style="border-radius: 12px; overflow: hidden; width: 240px; height: 260px;">
      <svg width="240" height="260" style="background: white">
        <g stroke="black" stroke-width="4" fill="none">
            <line x1="20" y1="235" x2="217" y2="235" />
            <line x1="60" y1="235" x2="60" y2="25" />
            <line x1="58" y1="25" x2="152" y2="25" />
            <line x1="150" y1="25" x2="150" y2="55" />
        </g>
        <g stroke="black" stroke-width="3" fill="none">{body}</g>
      </svg>
    </div>"""


def refresh(reveal=False):
    shown = " ".join(c if (c in state["guessed"] or reveal) else "_" for c in state["word"])
    word_html.value = f"<h2 style='{TEXT} letter-spacing: 4px; margin: 0;'>{shown}</h2>"
    drawing.value = draw(state["wrong"])
    if not state["over"]:
        status_html.value = f"<h3 style='{TEXT}'>Wrong guesses: {state['wrong']}/{MAX_WRONG}</h3>"


def end_game(message):
    state["over"] = True
    for b in letter_buttons.values():
        b.disabled = True
    status_html.value = f"<h3 style='{TEXT}'>{message}</h3>"


def guess(letter):
    if state["over"] or letter in state["guessed"]:
        return
    state["guessed"].add(letter)
    letter_buttons[letter].disabled = True
    if letter not in state["word"]:
        state["wrong"] += 1
        refresh()
        if state["wrong"] >= MAX_WRONG:
            refresh(reveal=True)
            end_game(f"💀 You lost! The word was {state['word']}.")
    else:
        refresh()
        if all(c in state["guessed"] for c in state["word"]):
            end_game(f"🎉 You won! The word was {state['word']}.")


def new_game(_=None):
    state.update(word=random.choice(WORDS).upper(), guessed=set(), wrong=0, over=False)
    for b in letter_buttons.values():
        b.disabled = False
    refresh()


for letter in string.ascii_uppercase:
    b = widgets.Button(description=letter, layout=widgets.Layout(width="48px"))
    b.on_click(lambda _, l=letter: guess(l))
    b.add_class("rounded-letter-btn")
    letter_buttons[letter] = b

new_btn.on_click(new_game)
new_btn.add_class("rounded-action-button")

keyboard = widgets.GridBox(
    list(letter_buttons.values()),
    layout=widgets.Layout(grid_template_columns="repeat(7, 52px)", margin="20px 0 0 0"),
)

new_game()

# Optional cosmetics: rounded corners (game still works if this CSS is ignored)
display(HTML("""
<style>
  .rounded-game-box { border-radius: 16px !important; background: #fafafa !important; }
  .rounded-action-button { border-radius: 6px !important; }
  .rounded-letter-btn { border-radius: 4px !important; }
</style>
"""))

game_card = widgets.VBox(
    [title_html, drawing, word_html, status_html, keyboard, new_btn],
    layout=widgets.Layout(
        align_items="center",
        padding="25px",
        border="1px solid #e0e0e0",
        width="460px",
        margin="20px 0 20px 0",
    ),
)
game_card.add_class("rounded-game-box")

display(game_card)
